# 10. Error Analysis

## Goal

Find where the final demo pipeline succeeds and where it still fails.

This notebook separates errors into ASR/cleaning, chunking, retrieval, and summary behavior.


## Why This Step Matters

Phase 10 proves the pipeline can answer one question. Error analysis checks whether that success is reliable across different question types.

The goal is not to fix every issue here. The goal is to identify the next highest-value fixes with evidence.


## Inputs

- The committed Phase 10 demo pipeline.
- A short Typhoon-like transcript from the manual meeting test.
- Several questions with expected answer terms.

This notebook does not call Whisper or the Typhoon API.


## Outputs

- A per-question pass/fail table.
- Retrieved context for each question.
- Observed failure source for each failed case.
- Small next-fix recommendations.


## Setup

Load the same final demo functions used by the CLI.


In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.pipeline.run_demo import _rows_from_transcript_text, load_config, run_demo_from_rows

CONFIG_PATH = PROJECT_ROOT / "configs" / "final_demo.yaml"
config = load_config(CONFIG_PATH)
config["top_k"] = 1
config["max_summary_points"] = 1

print("Project root:", PROJECT_ROOT)
print("ASR backend in config:", config["asr_backend"])
print("max_chunk_characters:", config["max_chunk_characters"])
print("top_k:", config["top_k"])


Project root: /home/lataeq/Code/Saranote
ASR backend in config: typhoon_api
max_chunk_characters: 70
top_k: 1


## Load Data

Use a Typhoon-like transcript from the manual audio test, then split it with the same API transcript splitter used by `run_demo.py`.


In [2]:
typhoon_like_transcript = (
    "สวัสดีครับ วันนี้เป็นการประชุมเรื่องแผนงบประมาณโครงการใหม่ "
    "ที่ประชุมมีมติให้ปรับแผนงบประมาณ โดยลดค่าใช้จ่าย "
    "ด้านอุปกรณ์และย้ายงบประมาณบางส่วน ไปสนับสนุน งานอบรมทีมงาน "
    "ฝ่ายดำเนินงานต้องส่งเอกสารงบประมาณฉบับแก้ไขภายในวันศุกร์ "
    "นอกจากนี้ ที่ประชุมยังพูดถึงความเสี่ยงเรื่องข้อมูลไม่ครบ และขอให้ทุก ฝ่าย "
    "ตรวจสอบข้อมูลก่อนส่งรายงาน"
)

rows = _rows_from_transcript_text(
    typhoon_like_transcript,
    max_characters=int(config["max_chunk_characters"]),
)

print(pd.DataFrame(rows).to_string(index=False))


    timestamp                                                                 text
transcript-01           สวัสดีครับ วันนี้เป็นการประชุมเรื่องแผนงบประมาณโครงการใหม่
transcript-02                     ที่ประชุมมีมติให้ปรับแผนงบประมาณ โดยลดค่าใช้จ่าย
transcript-03           ด้านอุปกรณ์และย้ายงบประมาณบางส่วน ไปสนับสนุน งานอบรมทีมงาน
transcript-04             ฝ่ายดำเนินงานต้องส่งเอกสารงบประมาณฉบับแก้ไขภายในวันศุกร์
transcript-05 นอกจากนี้ ที่ประชุมยังพูดถึงความเสี่ยงเรื่องข้อมูลไม่ครบ และขอให้ทุก
transcript-06                                      ฝ่าย ตรวจสอบข้อมูลก่อนส่งรายงาน


## Run Experiment

Each case has expected terms and optional forbidden terms. A failed case is useful here because it shows which part of the pipeline needs attention next.


In [3]:
cases = [
    {
        "case_id": "action_item",
        "question": "ฝ่ายดำเนินงานต้องทำอะไร",
        "expected_terms": ["ส่งเอกสารงบประมาณฉบับแก้ไข", "วันศุกร์"],
        "forbidden_terms": ["นอกจากนี้"],
        "expected_no_evidence": False,
        "expected_issue": "none",
    },
    {
        "case_id": "budget_decision_full",
        "question": "ที่ประชุมมีมติเรื่องงบประมาณอย่างไร",
        "expected_terms": ["ปรับแผนงบประมาณ", "ลดค่าใช้จ่าย", "ย้ายงบประมาณ"],
        "forbidden_terms": [],
        "expected_no_evidence": False,
        "expected_issue": "retrieval/chunking may need adjacent context",
    },
    {
        "case_id": "reallocation_target",
        "question": "ย้ายงบประมาณไปสนับสนุนอะไร",
        "expected_terms": ["งานอบรมทีมงาน"],
        "forbidden_terms": [],
        "expected_no_evidence": False,
        "expected_issue": "none",
    },
    {
        "case_id": "report_followup",
        "question": "ทุกฝ่ายต้องทำอะไรก่อนส่งรายงาน",
        "expected_terms": ["ทุกฝ่าย", "ตรวจสอบข้อมูลก่อนส่งรายงาน"],
        "forbidden_terms": [],
        "expected_no_evidence": False,
        "expected_issue": "chunking can split ทุก ฝ่าย across rows",
    },
    {
        "case_id": "unsupported_approver",
        "question": "ใครเป็นผู้อนุมัติโครงการนี้",
        "expected_terms": [],
        "forbidden_terms": [],
        "expected_no_evidence": True,
        "expected_issue": "retrieval false positive",
    },
]

case_overview = pd.DataFrame(
    {
        "case_id": case["case_id"],
        "question": case["question"],
        "expected_issue": case["expected_issue"],
    }
    for case in cases
)
print(case_overview.to_string(index=False))


             case_id                            question                               expected_issue
         action_item             ฝ่ายดำเนินงานต้องทำอะไร                                         none
budget_decision_full ที่ประชุมมีมติเรื่องงบประมาณอย่างไร retrieval/chunking may need adjacent context
 reallocation_target          ย้ายงบประมาณไปสนับสนุนอะไร                                         none
     report_followup      ทุกฝ่ายต้องทำอะไรก่อนส่งรายงาน      chunking can split ทุก ฝ่าย across rows
unsupported_approver         ใครเป็นผู้อนุมัติโครงการนี้                     retrieval false positive


## Inspect Results

Run every question through the same final demo function and classify the observed failure source.


In [4]:
def classify_case(case, result):
    answer = result["answer"]
    context = "\n".join(item["text"] for item in result["retrieved_context"])

    if case["expected_no_evidence"]:
        passed = "ไม่พบข้อมูล" in answer
    else:
        has_expected_terms = all(term in answer for term in case["expected_terms"])
        avoids_forbidden_terms = all(term not in answer for term in case["forbidden_terms"])
        passed = bool(result["retrieved_context"]) and has_expected_terms and avoids_forbidden_terms

    if passed:
        observed_issue = "none"
    elif case["expected_no_evidence"] and result["retrieved_context"]:
        observed_issue = "retrieval false positive"
    elif not all(term in context for term in case["expected_terms"]):
        observed_issue = "retrieval/chunking missed needed context"
    elif not all(term in answer for term in case["expected_terms"]):
        observed_issue = "summary missed expected term"
    elif any(term in answer for term in case["forbidden_terms"]):
        observed_issue = "summary included extra context"
    else:
        observed_issue = "unknown"

    return passed, observed_issue

records = []
context_records = []

for case in cases:
    result = run_demo_from_rows(rows, case["question"], config)
    passed, observed_issue = classify_case(case, result)
    context_text = "\n".join(item["text"] for item in result["retrieved_context"])

    records.append(
        {
            "case_id": case["case_id"],
            "passed": passed,
            "observed_issue": observed_issue,
            "answer": result["answer"],
        }
    )
    context_records.append(
        {
            "case_id": case["case_id"],
            "question": case["question"],
            "retrieved_context": context_text,
        }
    )

analysis_df = pd.DataFrame(records)
context_df = pd.DataFrame(context_records)

print(analysis_df[["case_id", "passed", "observed_issue", "answer"]].to_string(index=False))


             case_id  passed                           observed_issue                                                                                   answer
         action_item    True                                     none   จาก transcript ที่เกี่ยวข้อง: ฝ่ายดำเนินงานต้องส่งเอกสารงบประมาณฉบับแก้ไขภายในวันศุกร์
budget_decision_full   False retrieval/chunking missed needed context           จาก transcript ที่เกี่ยวข้อง: ที่ประชุมมีมติให้ปรับแผนงบประมาณ โดยลดค่าใช้จ่าย
 reallocation_target    True                                     none จาก transcript ที่เกี่ยวข้อง: ด้านอุปกรณ์และย้ายงบประมาณบางส่วน ไปสนับสนุน งานอบรมทีมงาน
     report_followup   False retrieval/chunking missed needed context                            จาก transcript ที่เกี่ยวข้อง: ฝ่าย ตรวจสอบข้อมูลก่อนส่งรายงาน
unsupported_approver   False                 retrieval false positive จาก transcript ที่เกี่ยวข้อง: สวัสดีครับ วันนี้เป็นการประชุมเรื่องแผนงบประมาณโครงการใหม่


## Retrieved Context

This table shows whether failures come from the selected transcript chunk or from the answer formatting step.


In [5]:
print(context_df.to_string(index=False))


             case_id                            question                                          retrieved_context
         action_item             ฝ่ายดำเนินงานต้องทำอะไร   ฝ่ายดำเนินงานต้องส่งเอกสารงบประมาณฉบับแก้ไขภายในวันศุกร์
budget_decision_full ที่ประชุมมีมติเรื่องงบประมาณอย่างไร           ที่ประชุมมีมติให้ปรับแผนงบประมาณ โดยลดค่าใช้จ่าย
 reallocation_target          ย้ายงบประมาณไปสนับสนุนอะไร ด้านอุปกรณ์และย้ายงบประมาณบางส่วน ไปสนับสนุน งานอบรมทีมงาน
     report_followup      ทุกฝ่ายต้องทำอะไรก่อนส่งรายงาน                            ฝ่าย ตรวจสอบข้อมูลก่อนส่งรายงาน
unsupported_approver         ใครเป็นผู้อนุมัติโครงการนี้ สวัสดีครับ วันนี้เป็นการประชุมเรื่องแผนงบประมาณโครงการใหม่


## What Failed

Count the observed issue types. These are the next engineering targets.


In [6]:
issue_counts = analysis_df["observed_issue"].value_counts().rename_axis("observed_issue").reset_index(name="count")
print(issue_counts.to_string(index=False))


                          observed_issue  count
                                    none      2
retrieval/chunking missed needed context      2
                retrieval false positive      1


## Next Fix Candidates

Keep the next fixes small and evidence-driven.


In [7]:
recommendations = pd.DataFrame(
    [
        {
            "priority": "high",
            "issue": "retrieval false positive",
            "candidate_fix": "Add a minimum retrieval score or no-evidence threshold.",
        },
        {
            "priority": "medium",
            "issue": "retrieval/chunking missed needed context",
            "candidate_fix": "Try adjacent-chunk expansion or top_k=2 for broad questions.",
        },
        {
            "priority": "low",
            "issue": "chunking can split ทุก ฝ่าย across rows",
            "candidate_fix": "Normalize repeated ASR phrase splits only if they recur in real tests.",
        },
    ]
)
print(recommendations.to_string(index=False))


priority                                    issue                                                          candidate_fix
    high                 retrieval false positive                Add a minimum retrieval score or no-evidence threshold.
  medium retrieval/chunking missed needed context           Try adjacent-chunk expansion or top_k=2 for broad questions.
     low  chunking can split ทุก ฝ่าย across rows Normalize repeated ASR phrase splits only if they recur in real tests.


## What We Learned

- The action-item query works after the Typhoon splitter separates `นอกจากนี้`.
- Broad budget questions can need evidence from more than one nearby chunk.
- Unsupported questions still need a no-evidence threshold.
- Some ASR spacing artifacts, such as `ทุก ฝ่าย`, can affect answer quality.


## Common Problems

- A single retrieved chunk can be too narrow for broad questions.
- N-gram fallback can retrieve related-looking text for unsupported questions.
- Fixing every ASR typo manually does not scale; only repeated domain mistakes should enter the tiny glossary.


## Next Step

Use this analysis to choose one focused fix: retrieval threshold first, or adjacent chunk expansion first.


## Suggested Commit

```bash
git add notebooks/10_error_analysis.ipynb
git commit -m "eval: add final pipeline error analysis"
```
